# Floor Analysis Verification — `FloorAnalysisPair`

Verifies `FloorAnalysis` and `FloorAnalysisPair` from
`scite/beam/floor/floor_analysis.py` against analytical references and
cross-checks with the beam-level `BeamDeflectionAnalysis`.

**Floor system taxonomy (European nomenclature)**
| Class | Section type | Reinforcement |
|---|---|---|
| **SRCFlatSlab** | Solid rectangular slab strip (1 m wide) | Steel (B500B) |
| **SRCRibbedSlab** | T-section rib + bay slab strip | Steel (B500B) |
| **CRCRibbedSlab** | T-section rib + bay slab strip | CFRP grid |

**Scope**
- Section 1 — SRC flat slab: elastic verification $w = 5qL^4/(384EI)$  
- Section 2 — SRC flat slab: p–w curve (surface load [kN/m²] vs midspan deflection)  
- Section 3 — SRC flat slab: span sweep $L$  
- Section 4 — SRC flat slab: reinforcement ratio $a_s$ sweep  
- Section 5 — SRC ribbed slab: rib vs bay-slab strip comparison  
- Section 6 — CRC ribbed slab: CFRP rib, linear-elastic response  
- Section 7 — SRC vs CRC ribbed slab: normalised comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scite.beam.floor import (
    FloorAnalysis, FloorAnalysisPair, BeamPair,
    FloorSectionRC, FloorSectionCRC,
    ec2_beff_rib,
    LoadModel, FlatSlab, SRCRibbedSlab, CRCRibbedSlab,
)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
BLUE, RED = '#1f77b4', '#d62728'
print('Imports OK')


---
## 1  SRC Flat Slab — elastic verification

For a uniformly distributed load on a simply supported beam:

$$
w_\mathrm{max} = \frac{5\, q\, L^4}{384\, E_c\, I}
$$

We use 5 % of the characteristic resistance load $q_R$ to stay in the
fully uncracked linear range and compare the numerical integration against
the analytical formula with $EI$ taken from the initial M–κ slope.

In [ ]:
# ── Reference SRC flat slab — 1 m-wide strip ─────────────────────────────────
# Typical intermediate floor:  h = 200 mm, a_s = 3 cm²/m → A_s = 300 mm²,
# c_s = 30 mm (concrete cover + bar radius), L = 5 m, f_ck = 30 MPa, f_yk = 500 MPa
b_ref  = 1000.0   # mm  (1 m-wide slab strip)
h_ref  = 200.0    # mm
A_s    = 300.0    # mm²  (≈ 3 cm²/m)
z_s    = 30.0     # mm   from bottom fibre
f_ck   = 30.0     # MPa
f_yk   = 500.0    # MPa
L_mm   = 5000.0   # mm  (5 m span)

pair_ref = FloorAnalysisPair.for_rc(
    b=b_ref, h=h_ref, f_ck=f_ck, A_s=A_s, z_s=z_s, f_yk=f_yk, L_mm=L_mm,
)
bda_sls = pair_ref.sls.bda
print(f'SLS — M_R = {bda_sls.M_R:.2f} kN·m,  q_R = {bda_sls.F_R:.3f} N/mm = {bda_sls.F_R:.2f} kN/m')

# Integration accuracy: EI from M-κ initial tangent
mk    = bda_sls.mk
n_fit = min(10, len(mk.M_values))
EI    = float(np.polyfit(mk.kappa_values[1:n_fit],
                          mk.M_values[1:n_fit] * 1e6, 1)[0])   # N·mm²

q_el        = 0.05 * bda_sls.F_R                              # N/mm  (5 % of q_R)
w_bda       = float(np.max(pair_ref.sls.get_w_x(q_el)))       # mm, positive downward
w_analytical = 5 * q_el * L_mm**4 / (384 * EI)                # mm

print()
print('Elastic verification (distributed load, 5 % of q_R)')
print('─' * 55)
print(f'q_el              = {q_el:.5f} N/mm  ({100*q_el/bda_sls.F_R:.0f} % of q_R)')
print(f'EI_eff  (M-κ)     = {EI:.3e} N·mm²')
print(f'w_BDA (integr.)   = {w_bda:.4f} mm')
print(f'w_analytic (M-κ)  = {w_analytical:.4f} mm  (5qL⁴/384EI, same EI_eff)')
print(f'Relative error    = {100*(w_bda - w_analytical)/w_analytical:.2f} %')

---
## 2  SRC Flat Slab — surface pressure p–w curve

`FloorAnalysis.get_pw(w_trib_m)` converts the internal force
$q$ [N/mm = kN/m] to surface pressure $p$ [kN/m²] via the tributary width:

$$
p \;[\text{kN/m}^2] = \frac{q \;[\text{kN/m}]}{w_{\mathrm{trib}} \;[\text{m}]}
$$

For the flat slab strip $w_{\mathrm{trib}} = 1$ m, so $p = q$ numerically.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

BLUE, RED = '#1f77b4', '#d62728'
w_sls = L_mm / 250.0   # SLS limit [mm]

# SLS (characteristic strengths)
p_sls, w_sls_arr = pair_ref.sls.get_pw(w_trib_m=1.0)
# ULS (design strengths)
p_uls, w_uls_arr = pair_ref.uls.get_pw(w_trib_m=1.0)

for ax in axes:
    ax.plot(w_sls_arr, p_sls, color=BLUE, lw=2.0, label='SLS (mean strengths)')
    ax.plot(w_uls_arr, p_uls, color=RED,  lw=2.0, linestyle='--', label='ULS (design strengths)')
    ax.axvline(w_sls, color='gray', linestyle=':', lw=1.5, label=f'L/250 = {w_sls:.0f} mm')
    ax.set_xlabel(r'$w_\mathrm{max}$ [mm]')
    ax.set_ylabel(r'$p$ [kN/m²]')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(fontsize=8)

axes[0].set_title(f'SRC Flat Slab  (b=1000, h={h_ref:.0f} mm, $a_s$={A_s/100:.1f} cm²/m, L={L_mm/1000:.1f} m)', fontsize=9)
axes[1].set_xlim(0, w_sls * 2)
axes[1].set_title('SLS region detail  (0 … 2·L/250)', fontsize=9)

plt.suptitle('Surface pressure vs midspan deflection — SRC Flat Slab', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3  SRC Flat Slab — span sweep

Longer spans are more prone to deflection-governed design.  The normalised
plot uses $w/L$ [‰] to remove the span-scaling effect and exposes the
structural efficiency of each span–depth combination.

In [ ]:
L_values = [3000.0, 4000.0, 5000.0, 6000.0, 8000.0]   # mm
colors   = plt.cm.viridis(np.linspace(0.1, 0.9, len(L_values)))

fig, (ax_abs, ax_norm) = plt.subplots(1, 2, figsize=(14, 5))

for L, color in zip(L_values, colors):
    pair = FloorAnalysisPair.for_rc(b=b_ref, h=h_ref, f_ck=f_ck,
                                     A_s=A_s, z_s=z_s, f_yk=f_yk, L_mm=L)
    p_arr, w_arr = pair.sls.get_pw(w_trib_m=1.0)
    lbl = f'L = {L/1000:.1f} m'
    ax_abs.plot(w_arr, p_arr, color=color, lw=1.8, label=lbl)
    ax_norm.plot(w_arr / L * 1e3, p_arr / pair.sls.bda.F_R,
                 color=color, lw=1.8, label=lbl)

ax_abs.axvline(h_ref * 2, color='red', ls='--', lw=1.2, alpha=0.7, label='h/100 ref.')
ax_abs.set_xlabel(r'$w_\mathrm{max}$ [mm]');  ax_abs.set_ylabel('p [kN/m²]')
ax_abs.set_title('Absolute p–w'); ax_abs.legend(fontsize=8); ax_abs.grid(True, alpha=0.3)

ax_norm.axvline(1/250*1e3, color='red', linestyle='--', lw=1.2, label='L/250')
ax_norm.set_xlabel(r'$w_\mathrm{max}/L$ [‰]'); ax_norm.set_ylabel(r'$q/q_R$  (SLS)')
ax_norm.set_title('Normalised p–w'); ax_norm.legend(fontsize=8); ax_norm.grid(True, alpha=0.3)

plt.suptitle(f'Span sweep — SRC Flat Slab  (h={h_ref:.0f} mm, $a_s$={A_s/100:.1f} cm²/m)',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 4  SRC Flat Slab — reinforcement ratio $a_s$ sweep

Higher $a_s$ raises $q_R$ (more moment capacity) and, for the same
service load, reduces the required slab depth.  The normalised plot
shows the relative deflection at a given fraction of $q_R$.

In [ ]:
a_s_values_cm2m = [1.0, 2.0, 3.0, 5.0, 8.0, 12.0]   # cm²/m
colors = plt.cm.plasma(np.linspace(0.1, 0.85, len(a_s_values_cm2m)))

fig, (ax_abs, ax_norm) = plt.subplots(1, 2, figsize=(14, 5))

for a_s, color in zip(a_s_values_cm2m, colors):
    A_s_mm2 = a_s * 100.0
    pair = FloorAnalysisPair.for_rc(b=b_ref, h=h_ref, f_ck=f_ck,
                                     A_s=A_s_mm2, z_s=z_s, f_yk=f_yk, L_mm=L_mm)
    p_arr, w_arr = pair.sls.get_pw(w_trib_m=1.0)
    q_R = pair.sls.bda.F_R
    fm  = pair.sls.bda.mk.get_failure_mode()
    lbl = f'$a_s$ = {a_s:.1f} cm²/m'
    ax_abs.plot(w_arr, p_arr, color=color, lw=1.8, label=lbl)
    ax_norm.plot(w_arr / L_mm * 1e3, p_arr / q_R, color=color, lw=1.8, label=lbl)
    print(f'a_s={a_s:5.1f} cm²/m  q_R={q_R:.3f} N/mm  mode={fm["mode"]:16s}  '
          f'η_c={fm["eta_c"]:.3f}  η_f={fm["eta_f"]:.3f}')

ax_abs.axvline(L_mm/250, color='red', ls='--', lw=1.2, label=f'L/250 = {L_mm/250:.0f} mm')
ax_abs.set_xlabel(r'$w_\mathrm{max}$ [mm]'); ax_abs.set_ylabel('p [kN/m²]')
ax_abs.set_title('Absolute p–w'); ax_abs.legend(fontsize=8); ax_abs.grid(True, alpha=0.3)

ax_norm.axvline(1/250*1e3, color='red', ls='--', lw=1.2, label='L/250')
ax_norm.set_xlabel(r'$w_\mathrm{max}/L$ [‰]'); ax_norm.set_ylabel(r'$q/q_R$  (SLS)')
ax_norm.set_title('Normalised p–w'); ax_norm.legend(fontsize=8); ax_norm.grid(True, alpha=0.3)

plt.suptitle(f'$a_s$ sweep — SRC Flat Slab  (h={h_ref:.0f} mm, L={L_mm/1000:.1f} m)',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 5  SRC Ribbed Slab — compositional floor-system model

The floor system is represented as a **`SRCRibbedSlab`** object that encapsulates
all geometry, section, and analysis sub-models:

| Component | Class / property |
|-----------|-----------------|
| Floor geometry | `H_rib`, `H_bay`, `B_rib`, `L_rib`, `L_bay` |
| EC2 §5.3.2.1 effective flange | `b_eff` (computed via `ec2_beff_rib`) |
| Rib beam analysis (SLS + ULS) | `rib_beam : BeamPair` |
| Bay slab analysis (SLS + ULS) | `bay_beam : BeamPair` |
| Self-weight | `g_k` property (web + bay slab, concrete-only) |
| Material volumes & GWP | `volumes()` dict |

The **`LoadModel`** holds the EN 1990 load combination parameters and computes:

$$
p_{Ed,qp} = g_{perm} + \psi_2\,q_k, \qquad
p_{Ed,u}  = \gamma_G\,g_{perm} + \gamma_Q\,q_k
$$

`plot_floor_assessment(axes, lm)` draws the three-panel view: load breakdown,
bay-slab p–w, and rib p–w — each with EC0 demand lines.


In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
lm = LoadModel(delta_g_k=2.0, q_k=5.0)

# ── SRC Ribbed Slab ───────────────────────────────────────────────────────────
src_slab = SRCRibbedSlab(
    H_rib=200, H_bay=50, B_rib=100, L_rib=6000, L_bay=600,
    A_s_rib=1100.0, z_s_rib=35.0, f_ck=30.0, f_yk=500.0,
    A_s_bay=150.0, z_s_bay=20.0,
)
src_slab.report(lm)
print()
# Expose variables used in downstream cells (sections 6 & 7)
pair_rib  = src_slab.rib_beam
pair_bay  = src_slab.bay_beam
pair_src  = src_slab.rib_beam   # canonical name for comparison plots
L_rib_mm  = src_slab.L_rib
L_slab_mm = src_slab.L_bay
s_m       = src_slab.s_m
b_w, h_w, b_f, h_f = src_slab.B_rib, src_slab.h_w, src_slab.b_eff, src_slab.H_bay

# ── Three-panel floor assessment ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
src_slab.plot_floor_assessment(axes, lm)
plt.suptitle('SRC Ribbed Slab — floor system assessment', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 6  CRC Ribbed Slab — CFRP rib

**`CRCRibbedSlab`** uses the same floor geometry as section 5 but replaces the
steel rib reinforcement with a CFRP grid.  Key structural differences:

- **No yielding plateau** — linear-elastic to brittle rupture at $\varepsilon_{fu} = f_{fk}/E_f$
- Much higher tensile strength $f_{fk} \approx 3000$ MPa vs $f_{yk} = 500$ MPa
- Lower reinforcement area per rib (thin grid geometry)
- Reinforcement-governed failure at ULS; partial factor $\gamma_f = 1.25$


In [ ]:
# ── CRC Ribbed Slab — same floor geometry as section 5, CFRP grid ────────────
# Solidian Q95-like: A_f ≈ 50.3 mm², z_f = 25 mm from web bottom
# E_f = 210 GPa,  f_fk = 3000 MPa
crc_slab = CRCRibbedSlab(
    H_rib=210, H_bay=50, B_rib=50, L_rib=6000, L_bay=1000,
    A_f_rib=500.3, z_f_rib=25.0, f_ck=30.0, E_f=210_000.0, f_fk=3000.0,
    A_s_bay=150.0, z_s_bay=10.0,
)
pair_crc = crc_slab.rib_beam   # used in section 7
crc_slab.report(lm)
print()

fm_crc = pair_crc.sls.bda.mk.get_failure_mode()
fm_src = pair_src.sls.bda.mk.get_failure_mode()
print(f'CRC rib: M_R={pair_crc.sls.bda.M_R:.2f} kN·m  mode={fm_crc["mode"]:16s}  '
      f'η_c={fm_crc["eta_c"]:.3f}  η_f={fm_crc["eta_f"]:.3f}')
print(f'SRC rib: M_R={pair_src.sls.bda.M_R:.2f} kN·m  mode={fm_src["mode"]:16s}  '
      f'η_c={fm_src["eta_c"]:.3f}  η_f={fm_src["eta_f"]:.3f}')
print()

# ── Three-panel floor assessment ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
crc_slab.plot_floor_assessment(axes, lm)
plt.suptitle('CRC Ribbed Slab — floor system assessment', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 7  Comparative assessment — SRC vs CRC Ribbed Slab

Left panel: **absolute** p–w curves with EC0 demand lines
(both systems share the same geometry → same $g_k$, same $p_{Ed,qp}$, $p_{Ed,u}$).

Right panel: **normalised** $p/p_R$ vs $w/L$ to compare capacity utilisation
at the L/250 serviceability limit.


In [ ]:
s = lm.surface_loads(src_slab.g_k)   # same g_k for both (identical geometry)

fig, (ax_abs, ax_norm) = plt.subplots(1, 2, figsize=(15, 5))

# ── Left: absolute p-w with demand lines ─────────────────────────────────────
for slab, pair, color, ls, lbl in [
    (src_slab, pair_src, BLUE,      '--', f'SRC rib — B500B  (M_R={pair_src.sls.bda.M_R:.1f} kN·m)'),
    (crc_slab, pair_crc, '#2ca02c', '-',  f'CRC rib — CFRP   (M_R={pair_crc.sls.bda.M_R:.1f} kN·m)'),
]:
    p_arr, w_arr = pair.sls.get_pw(slab.s_m)
    ax_abs.plot(w_arr, p_arr, color=color, lw=2, ls=ls, label=lbl)

# Demand lines
ax_abs.axhspan(0, s['p_Ed_u'],  color=RED,  alpha=0.08, zorder=0)
ax_abs.axhline(s['p_Ed_qp'], color=BLUE, lw=1.6,
               label=fr"$p_{{Ed,qp}}$ = {s['p_Ed_qp']:.2f} kN/m²  (SLS)")
ax_abs.axhline(s['p_Ed_u'],  color=RED,  lw=1.6,
               label=fr"$p_{{Ed,u}}$  = {s['p_Ed_u']:.2f} kN/m²  (ULS)")
ax_abs.axvspan(0, L_rib_mm/250, color=BLUE, alpha=0.08, zorder=0)
ax_abs.axvline(L_rib_mm/250, color=BLUE, ls='--', lw=1.5,
               label=f'L/250 = {L_rib_mm/250:.0f} mm')
ax_abs.set_xlabel(r'$w_\mathrm{max}$ [mm]')
ax_abs.set_ylabel('p [kN/m²]')
ax_abs.set_title('Absolute p–w  (SLS capacities + EC0 demands)', fontsize=9)
ax_abs.legend(fontsize=7, loc='lower right')
ax_abs.grid(True, linestyle='--', alpha=0.4)

# ── Right: normalised p/p_R vs w/L ───────────────────────────────────────────
for slab, pair, color, ls, lbl in [
    (src_slab, pair_src, BLUE,      '--', 'SRC rib — B500B (SLS)'),
    (crc_slab, pair_crc, '#2ca02c', '-',  'CRC rib — CFRP  (SLS)'),
]:
    p_arr, w_arr = pair.sls.get_pw(slab.s_m)
    p_R = pair.sls.bda.F_R / slab.s_m
    ax_norm.plot(w_arr / L_rib_mm * 1e3, p_arr / p_R,
                 color=color, lw=2, ls=ls, label=lbl)

ax_norm.axvline(1/250*1e3, color='red', ls='--', lw=1.5, label='L/250')
ax_norm.set_xlabel(r'$w_\mathrm{max}/L$ [‰]')
ax_norm.set_ylabel(r'$p / p_{R,\mathrm{SLS}}$')
ax_norm.set_title('Normalised p/p_R vs w/L', fontsize=9)
ax_norm.legend(fontsize=9)
ax_norm.grid(True, alpha=0.3)

plt.suptitle(
    f'SRC vs CRC Ribbed Slab — rib comparison  '
    f'(b_w={b_w:.0f}, h_w={h_w:.0f}, b_f={b_f:.0f}, h_f={h_f:.0f} mm,  '
    f'L={L_rib_mm/1e3:.1f} m, s={s_m:.2f} m)',
    fontsize=10, fontweight='bold',
)
plt.tight_layout(); plt.show()

# ── Capacity utilisation at SLS limit ────────────────────────────────────────
print(f"{'System':<8}  {'p(L/250)':>10}  {'p_R':>8}  {'η = p/p_R':>10}  "
      f"{'p_Ed,qp':>10}  {'η_qp':>8}")
print('-' * 60)
for slab, pair, lbl in [
    (src_slab, pair_src, 'SRC'),
    (crc_slab, pair_crc, 'CRC'),
]:
    p_arr, w_arr = pair.sls.get_pw(slab.s_m)
    p_R      = pair.sls.bda.F_R / slab.s_m
    p_at_sls = float(np.interp(L_rib_mm/250, w_arr, p_arr))
    p_qp     = s['p_Ed_qp']
    print(f'{lbl:<8}  {p_at_sls:>10.2f}  {p_R:>8.2f}  '
          f'{100*p_at_sls/p_R:>9.1f}%  '
          f'{p_qp:>10.2f}  {100*p_qp/p_R:>7.1f}%')
